# ✉️ AI-EMAIL-GENERATOR
> Generate · Edit · Send via SMTP · Save History

---

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install anthropic ipywidgets --quiet

## ⚙️ Step 2 — Imports

In [ ]:
import anthropic, json, smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from datetime import datetime
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
print('✅ Imports OK')

## 🔑 Step 3 — Anthropic API Key

In [ ]:
api_key_widget = widgets.Password(
    placeholder='sk-ant-api03-...',
    description='API Key:',
    layout=widgets.Layout(width='500px')
)
display(HTML('<h4 style="color:#818cf8">Enter your Anthropic API key</h4>'))
display(api_key_widget)
display(HTML('<small>Get yours at: <a href="https://console.anthropic.com/settings/keys" target="_blank">console.anthropic.com</a></small>'))

## 🧠 Step 4 — Core Functions (Run This Cell)

In [ ]:
HISTORY_FILE = Path('email_history.json')

ENRON_SAMPLES = [
    {'subject': 'Meeting Request', 'body': 'Greg,\n\nHow about next Tuesday or Thursday?\n\nPhillip'},
    {'subject': 'Business Trip',   'body': 'Traveling to have a business meeting takes the fun out of the trip. I would suggest holding the meeting here.'},
    {'subject': 'Re: Internet',    'body': '1. login: pallen\n\nI do not think these are required by the ISP.'},
]

def load_history():
    if HISTORY_FILE.exists():
        try: return json.loads(HISTORY_FILE.read_text())
        except: pass
    return []

def save_history(h): HISTORY_FILE.write_text(json.dumps(h, indent=2))

def add_to_history(entry):
    h = load_history()
    h.insert(0, {**entry, 'id': datetime.now().isoformat(),
                 'saved_at': datetime.now().strftime('%b %d, %Y %H:%M')})
    save_history(h[:50])

def generate_email(prompt, email_type, tone, recipient='', sender=''):
    api_key = api_key_widget.value.strip()
    if not api_key: raise ValueError('API key empty — fill in Step 3!')
    client = anthropic.Anthropic(api_key=api_key)
    examples = '\n---\n'.join(f"Subject: {e['subject']}\n{e['body']}" for e in ENRON_SAMPLES)
    system = f"""You are an expert email writer. Style examples from Enron dataset:\n{examples}\n\nRULES:\n- Tone: {tone}\n- Email type: {email_type}\n{('- To: '+recipient) if recipient else ''}\n{('- From: '+sender) if sender else ''}\n- Return ONLY valid JSON: {{\"subject\": \"...\", \"body\": \"...\"}}"""
    msg = client.messages.create(
        model='claude-sonnet-4-6', max_tokens=1000, system=system,
        messages=[{'role': 'user', 'content': f'Write an email: {prompt}'}]
    )
    text = msg.content[0].text.strip().replace('```json','').replace('```','').strip()
    return json.loads(text)

def send_smtp(subject, body, to_addr, host, port, user, password, from_name=''):
    msg = MIMEMultipart()
    msg['From']    = f'{from_name} <{user}>' if from_name else user
    msg['To']      = to_addr
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    with smtplib.SMTP(host, int(port)) as s:
        s.starttls()
        s.login(user, password)
        s.sendmail(user, to_addr, msg.as_string())
    return '✅ Email sent!'

print('✅ Functions ready!')

## ✦ Step 5 — Generate Email (Interactive UI)

In [ ]:
# Widget layouts
W500 = widgets.Layout(width='500px')
W300 = widgets.Layout(width='300px')

display(HTML("""
<style>
.success{background:rgba(16,185,129,.12);border:1px solid rgba(16,185,129,.3);border-radius:8px;padding:10px 14px;color:#34d399;margin:6px 0}
.error  {background:rgba(239,68,68,.12); border:1px solid rgba(239,68,68,.3); border-radius:8px;padding:10px 14px;color:#f87171;margin:6px 0}
.rbox   {background:rgba(99,102,241,.08);border:1px solid rgba(99,102,241,.3);border-radius:12px;padding:16px;margin:10px 0}
</style>
"""))

w_prompt    = widgets.Textarea(placeholder='E.g. Write a follow-up about my proposal sent last week...', layout=widgets.Layout(width='600px', height='90px'))
w_type      = widgets.Dropdown(options=['Business Proposal','Follow-up','Introduction','Request','Thank You','Meeting Invite','Cold Outreach','Complaint','Custom'], value='Follow-up', description='Type:', style={'description_width':'60px'}, layout=W300)
w_tone      = widgets.Dropdown(options=['Professional','Friendly','Formal','Concise','Persuasive','Apologetic'], value='Professional', description='Tone:', style={'description_width':'60px'}, layout=W300)
w_sender    = widgets.Text(placeholder='your@email.com',    description='From:', style={'description_width':'60px'}, layout=W500)
w_recipient = widgets.Text(placeholder='them@email.com',    description='To:',   style={'description_width':'60px'}, layout=W500)

btn_gen   = widgets.Button(description='✦ Generate Email', layout=widgets.Layout(width='200px', height='40px'), style={'button_color':'#6366f1','font_weight':'bold'})
btn_clear = widgets.Button(description='Clear', button_style='warning', layout=widgets.Layout(width='100px', height='40px'))
out_gen   = widgets.Output()

w_subj = widgets.Text(description='Subject:', style={'description_width':'80px'}, layout=widgets.Layout(width='600px'))
w_body = widgets.Textarea(description='Body:', style={'description_width':'80px'}, layout=widgets.Layout(width='600px', height='200px'))
btn_save = widgets.Button(description='💾 Save', button_style='success', layout=widgets.Layout(width='130px'))
btn_show = widgets.Button(description='📋 Show', button_style='info',    layout=widgets.Layout(width='130px'))
out_act  = widgets.Output()

edit_section = widgets.VBox([
    widgets.HTML('<hr><h4 style="color:#818cf8">✏️ Edit Your Email</h4>'),
    w_subj, w_body,
    widgets.HTML('<h4 style="color:#818cf8">Actions</h4>'),
    widgets.HBox([btn_save, btn_show]),
    out_act
])
edit_section.layout.display = 'none'

def on_gen(b):
    with out_gen:
        clear_output(wait=True)
        if not w_prompt.value.strip():
            display(HTML('<div class="error">⚠️ Describe what you want to write.</div>')); return
        display(HTML('<div style="color:#818cf8">⏳ Claude is writing your email…</div>'))
    try:
        r = generate_email(w_prompt.value, w_type.value, w_tone.value, w_recipient.value, w_sender.value)
        w_subj.value = r.get('subject','')
        w_body.value = r.get('body','')
        edit_section.layout.display = ''
        with out_gen: clear_output(wait=True); display(HTML('<div class="success">✅ Email generated! Edit below.</div>'))
    except Exception as e:
        with out_gen: clear_output(wait=True); display(HTML(f'<div class="error">❌ {e}</div>'))

def on_clear(b):
    w_prompt.value=w_subj.value=w_body.value=''
    edit_section.layout.display='none'
    with out_gen: clear_output()

def on_save(b):
    add_to_history({'subject':w_subj.value,'body':w_body.value,'prompt':w_prompt.value,'tone':w_tone.value,'email_type':w_type.value,'recipient':w_recipient.value,'sender':w_sender.value})
    with out_act: clear_output(wait=True); display(HTML('<div class="success">✅ Saved to email_history.json</div>'))

def on_show(b):
    with out_act:
        clear_output(wait=True)
        display(HTML(f'<div class="rbox"><b>Subject:</b> {w_subj.value}<hr><pre style="white-space:pre-wrap;color:#cbd5e1">{w_body.value}</pre></div>'))

btn_gen.on_click(on_gen)
btn_clear.on_click(on_clear)
btn_save.on_click(on_save)
btn_show.on_click(on_show)

display(HTML('<h3 style="color:#818cf8">✦ Compose</h3>'))
display(w_prompt)
display(widgets.HBox([w_type, w_tone]))
display(w_sender, w_recipient)
display(widgets.HBox([btn_gen, btn_clear]))
display(out_gen)
display(edit_section)

## 📨 Step 6 — Send via SMTP

In [ ]:
W400 = widgets.Layout(width='420px')
DS   = {'description_width': '110px'}

s_host = widgets.Text(value='smtp.gmail.com', description='SMTP Host:', style=DS, layout=W400)
s_port = widgets.Text(value='587',            description='Port:',      style=DS, layout=W400)
s_user = widgets.Text(placeholder='you@gmail.com', description='Username:', style=DS, layout=W400)
s_pass = widgets.Password(placeholder='App Password', description='Password:', style=DS, layout=W400)
s_name = widgets.Text(placeholder='Your Name', description='Display Name:', style=DS, layout=W400)
s_to   = widgets.Text(placeholder='recipient@email.com', description='Send To:', style=DS, layout=W400)

btn_send = widgets.Button(description='📨 Send Email Now', layout=widgets.Layout(width='200px', height='40px'), style={'button_color':'#6366f1','font_weight':'bold'})
out_smtp = widgets.Output()

def on_send(b):
    with out_smtp:
        clear_output(wait=True)
        to = s_to.value.strip() or w_recipient.value.strip()
        if not to:            display(HTML('<div class="error">❌ Enter recipient email.</div>')); return
        if not s_user.value:  display(HTML('<div class="error">❌ Enter SMTP username.</div>')); return
        if not s_pass.value:  display(HTML('<div class="error">❌ Enter SMTP password.</div>')); return
        display(HTML('<div style="color:#818cf8">📨 Sending…</div>'))
        try:
            r = send_smtp(w_subj.value or '(No subject)', w_body.value or '(No body)',
                          to, s_host.value, s_port.value, s_user.value, s_pass.value, s_name.value)
            display(HTML(f'<div class="success">{r}</div>'))
        except Exception as e:
            display(HTML(f'<div class="error">❌ {e}</div>'))

btn_send.on_click(on_send)

display(HTML('<h3 style="color:#818cf8">📨 SMTP Settings</h3>'))
display(HTML('<div style="background:rgba(245,158,11,.1);border:1px solid rgba(245,158,11,.3);border-radius:8px;padding:10px 14px;color:#fcd34d;font-size:13px;margin-bottom:10px">💡 <b>Gmail:</b> Enable 2-Step Verification → <a href="https://myaccount.google.com/apppasswords" target="_blank" style="color:#fcd34d">generate App Password</a> → use it as the password below.</div>'))
display(s_host, s_port, s_user, s_pass, s_name, s_to, btn_send, out_smtp)

## 📂 Step 7 — Email History

In [ ]:
btn_refresh  = widgets.Button(description='🔄 Refresh', button_style='info',   layout=widgets.Layout(width='140px'))
btn_clr_hist = widgets.Button(description='🗑️ Clear All', button_style='danger', layout=widgets.Layout(width='140px'))
out_hist = widgets.Output()

def show_history(b=None):
    with out_hist:
        clear_output(wait=True)
        hist = load_history()
        if not hist:
            display(HTML('<p style="color:#64748b">No saved emails yet.</p>')); return
        display(HTML(f'<p style="color:#94a3b8"><b>{len(hist)}</b> email(s) saved</p>'))
        for i, item in enumerate(hist):
            body_preview = item.get('body','')[:180]
            display(HTML(f"""
            <div style="background:rgba(255,255,255,0.03);border:1px solid rgba(255,255,255,0.08);border-radius:10px;padding:12px 16px;margin-bottom:8px">
              <div style="color:#e2e8f0;font-weight:600">{i+1}. {item.get('subject','(No subject)')}</div>
              <div style="color:#64748b;font-size:12px;margin:3px 0">{item.get('email_type','')} · {item.get('tone','')} · {item.get('saved_at','')}</div>
              <div style="color:#94a3b8;font-size:13px">{body_preview}{'…' if len(item.get('body',''))>180 else ''}</div>
            </div>"""))

def clear_all(b):
    save_history([])
    with out_hist: clear_output(wait=True); display(HTML('<div class="success">✅ History cleared.</div>'))

btn_refresh.on_click(show_history)
btn_clr_hist.on_click(clear_all)

display(HTML('<h3 style="color:#818cf8">📂 Saved Emails</h3>'))
display(widgets.HBox([btn_refresh, btn_clr_hist]))
display(out_hist)
show_history()

## 🐍 Step 8 — Direct Python Call (No UI)
You can also call `generate_email()` directly without the widget UI:

In [ ]:
# Make sure you ran Steps 2-4 first and entered your API key in Step 3

result = generate_email(
    prompt     = 'Write a follow-up email to a client about the proposal I sent last week',
    email_type = 'Follow-up',
    tone       = 'Professional',
    recipient  = 'client@company.com',
    sender     = 'me@mycompany.com',
)

print('📧 SUBJECT:', result['subject'])
print()
print('📝 BODY:')
print(result['body'])